# SMS Spam Detection

This notebook demonstrates a complete SMS spam classification workflow using local project files, reproducible preprocessing, and model comparison.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt


In [ ]:
DATA_PATH = Path('data/raw/spam.csv')
DATA_PATH.exists()


In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

df = pd.read_csv(DATA_PATH, encoding='latin-1').iloc[:, :2].copy()
df.columns = ['label', 'message']
df['message'] = df['message'].fillna('').map(clean_text)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()


In [ ]:
df['label'].value_counts()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['message'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label'],
)

def build_models():
    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )
    return {
        'MultinomialNB': Pipeline([('vectorizer', vectorizer), ('classifier', MultinomialNB())]),
        'LogisticRegression': Pipeline([('vectorizer', vectorizer), ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced'))]),
        'LinearSVC': Pipeline([('vectorizer', vectorizer), ('classifier', LinearSVC(class_weight='balanced'))]),
    }


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = {}

for name, model in build_models().items():
    score = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1').mean()
    scores[name] = score

scores


In [ ]:
best_model_name = max(scores, key=scores.get)
best_model = build_models()[best_model_name]
best_model.fit(X_train, y_train)
predictions = best_model.predict(X_test)

best_model_name


In [ ]:
print(classification_report(y_test, predictions, target_names=['ham', 'spam']))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=['ham', 'spam'],
    cmap='Blues',
    colorbar=False,
)
plt.title('SMS Spam Detection Confusion Matrix')
plt.show()


In [ ]:
sample_message = 'Congratulations! You have won a free trip. Call now.'
predicted_label = best_model.predict([sample_message])[0]
'spam' if predicted_label == 1 else 'ham'
